In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, accuracy_score
)
import matplotlib.pyplot as plt

# ── Load observations ──
observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')
print(f"Total observations: {len(observations)}")
print(f"Stars: {observations['star_name'].nunique()}")

## Compute per-star summary statistics

Same 16 physical features as `baseline.ipynb` plus 2 activity-variance flags (18 total). No observational proxies.

In [ ]:
def compute_star_features(group):
    """Compute summary statistics for one star from its observation sequence."""
    features = {}

    # -- RV statistics --
    features['rv_std'] = group['rv_centered'].std()
    features['rv_range'] = group['rv_centered'].max() - group['rv_centered'].min()
    features['rv_mean_abs_dev'] = (group['rv_centered'] - group['rv_centered'].median()).abs().mean()
    features['rv_skew'] = group['rv_centered'].skew()
    features['rv_kurtosis'] = group['rv_centered'].kurtosis()

    # -- RV error statistics --
    features['rv_err_mean'] = group['rv_err'].mean()
    features['rv_err_std'] = group['rv_err'].std()

    # -- Activity indicator statistics --
    features['rhkp_std'] = group['RHKp'].std()
    features['rhkp_range'] = group['RHKp'].max() - group['RHKp'].min()
    features['rhkp_mean'] = group['RHKp'].mean()
    features['halpha_std'] = group['Halpha'].std()
    features['halpha_range'] = group['Halpha'].max() - group['Halpha'].min()
    features['halpha_mean'] = group['Halpha'].mean()

    # -- RV-activity correlations --
    features['rv_rhkp_corr'] = group['rv_centered'].corr(group['RHKp'])
    features['rv_halpha_corr'] = group['rv_centered'].corr(group['Halpha'])
    features['rhkp_halpha_corr'] = group['RHKp'].corr(group['Halpha'])

    # -- Binary variance flags: 1 if indicator had variance, 0 if constant (corr NaN -> aliased to 0) --
    features['rhkp_has_variance'] = 1 if group['RHKp'].std() > 0 else 0
    features['halpha_has_variance'] = 1 if group['Halpha'].std() > 0 else 0

    # -- Label --
    features['has_exoplanets'] = group['has_exoplanets'].iloc[0]

    return pd.Series(features)

star_features = observations.groupby('star_name').apply(compute_star_features, include_groups=False).reset_index()
star_features = star_features.fillna(0)
star_features['has_exoplanets'] = star_features['has_exoplanets'].astype(int)

physical_features = [
    'rv_std', 'rv_range', 'rv_mean_abs_dev', 'rv_skew', 'rv_kurtosis',
    'rv_err_mean', 'rv_err_std',
    'rhkp_std', 'rhkp_range', 'rhkp_mean',
    'halpha_std', 'halpha_range', 'halpha_mean',
    'rv_rhkp_corr', 'rv_halpha_corr', 'rhkp_halpha_corr',
    'rhkp_has_variance', 'halpha_has_variance'
]

print(f"Stars: {len(star_features)}")
print(f"Positive: {star_features['has_exoplanets'].sum()}")
print(f"Negative: {(star_features['has_exoplanets'] == 0).sum()}")
print(f"Features ({len(physical_features)}): {physical_features}")

## Multi-Seed Evaluation

For each seed in `range(42, 48)`:
1. Build a stratified 60/20/20 split (same two-stage logic as `split.py`, but with `random_state=seed`)
2. No standardization (RF splits are invariant to monotonic feature transforms)
3. Train `RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=seed)`
4. Choose F1-optimal threshold on the validation set
5. Evaluate on the test set: ROC-AUC, PR-AUC, F1, precision, recall, accuracy
6. Record feature importances

After the loop: report per-seed results, mean ± std, and feature importance stability.

In [ ]:
SEEDS = list(range(43, 48))

results = {
    'seed': [], 'roc_auc': [], 'pr_auc': [], 'f1': [],
    'precision': [], 'recall': [], 'accuracy': [], 'threshold': [],
    'ci_lo': [], 'ci_hi': [], 'f05': [],
}
importances_per_seed = {}

from split import bootstrap_roc_auc
from sklearn.metrics import f1_score as sk_f1_score, fbeta_score

X_mat = star_features[physical_features].values
y_vec = star_features['has_exoplanets'].values.astype(int)

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"SEED {seed}")
    print(f"{'='*60}")

    # ── 1. Stratified split (two-stage, same as split.py) ──
    idx = np.arange(len(star_features))
    idx_train, idx_temp, y_train_split, y_temp = train_test_split(
        idx, y_vec, test_size=0.4, random_state=seed, stratify=y_vec,
    )
    idx_val, idx_test, y_val_split, y_test_split = train_test_split(
        idx_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp,
    )

    X_train, y_train = X_mat[idx_train], y_vec[idx_train]
    X_val,   y_val   = X_mat[idx_val],   y_vec[idx_val]
    X_test,  y_test  = X_mat[idx_test],  y_vec[idx_test]

    # No standardization: RF splits are invariant to monotonic feature transforms,
    # so (X - mean)/std would not change a single tree split. Omitted for clarity.
    X_train_s, X_val_s, X_test_s = X_train, X_val, X_test

    print(f"Train: {len(X_train_s)} (pos={y_train.sum()}, neg={(1-y_train).sum()})")
    print(f"Val:   {len(X_val_s)} (pos={y_val.sum()}, neg={(1-y_val).sum()})")
    print(f"Test:  {len(X_test_s)} (pos={y_test.sum()}, neg={(1-y_test).sum()})")

    # ── 3. Train RF ──
    rf = RandomForestClassifier(
        n_estimators=500, class_weight='balanced',
        random_state=seed, max_depth=None,
    )
    rf.fit(X_train_s, y_train)

    # ── 4. Threshold on VAL ──
    val_probs = rf.predict_proba(X_val_s)[:, 1]
    val_prec, val_rec, val_thr = precision_recall_curve(y_val, val_probs)
    val_f1 = 2 * val_prec * val_rec / (val_prec + val_rec + 1e-8)
    best_idx = int(np.argmax(val_f1))
    threshold = float(val_thr[best_idx]) if best_idx < len(val_thr) else 0.5
    val_f1_best = sk_f1_score(y_val, (val_probs >= threshold).astype(int), zero_division=0)

    # ── 5. Evaluate on TEST ──
    test_probs = rf.predict_proba(X_test_s)[:, 1]
    roc = roc_auc_score(y_test, test_probs)
    pr  = average_precision_score(y_test, test_probs)

    test_preds = (test_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, test_preds)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = sk_f1_score(y_test, test_preds, zero_division=0)
    f05       = fbeta_score(y_test, test_preds, beta=0.5, zero_division=0)
    acc       = accuracy_score(y_test, test_preds)

    results['seed'].append(seed)
    results['roc_auc'].append(roc)
    results['pr_auc'].append(pr)
    results['f1'].append(f1)
    results['f05'].append(f05)
    results['precision'].append(precision)
    results['recall'].append(recall)
    results['accuracy'].append(acc)
    results['threshold'].append(threshold)
    importances_per_seed[seed] = rf.feature_importances_

    print(f"ROC-AUC:  {roc:.4f}")
    print(f"PR-AUC:   {pr:.4f}")
    print(f"F1   (val→test, thr={threshold:.4f}): {f1:.4f}  (val F1={val_f1_best:.4f})")
    print(f"F0.5 (val→test, thr={threshold:.4f}): {f05:.4f}")
    print(f"Precision: {precision:.4f}  Recall: {recall:.4f}  Accuracy: {acc:.4f}")
    print(f"Confusion: TP={tp} FP={fp} TN={tn} FN={fn}")

    # Bootstrap 95% CI for ROC-AUC on this seed's test set
    ci_point, ci_lo, ci_hi = bootstrap_roc_auc(y_test, test_probs, n_resamples=200, seed=seed)
    print(f"  Bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    results['ci_lo'].append(ci_lo)
    results['ci_hi'].append(ci_hi)

print(f"\n\n{'='*60}")
print("ALL SEEDS COMPLETE")
print(f"{'='*60}")

## Summary: Mean ± Std Across Seeds

In [ ]:
results_df = pd.DataFrame(results)
print(results_df[['seed', 'roc_auc', 'ci_lo', 'ci_hi', 'pr_auc', 'f1', 'f05', 'precision', 'recall', 'accuracy', 'threshold']].to_string(index=False))

print(f"\n{'='*60}")
print(f"AGGREGATE STATISTICS (mean ± std, n={len(SEEDS)} seeds)")
print(f"{'='*60}")
for metric in ['roc_auc', 'pr_auc', 'f1', 'f05', 'precision', 'recall', 'accuracy', 'threshold']:
    vals = np.array(results[metric])
    print(f"  {metric:12s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}  (min={vals.min():.4f}, max={vals.max():.4f})")
print(f"\n  Bootstrap CIs per seed:")
for i, s in enumerate(SEEDS):
    print(f"    Seed {s}: ROC-AUC {results['roc_auc'][i]:.4f} [{results['ci_lo'][i]:.4f}, {results['ci_hi'][i]:.4f}]")

## Feature Importance Stability

How consistent are the feature importances across seeds? If the ranking shuffles wildly,
the model is latching onto noise. If the top features are stable, the RF is learning
the same underlying signal regardless of split.

In [ ]:
imp_df = pd.DataFrame(importances_per_seed, index=physical_features)
imp_df['mean'] = imp_df.mean(axis=1)
imp_df['std']  = imp_df.std(axis=1, ddof=1)
imp_df['cv']   = imp_df['std'] / imp_df['mean'].clip(lower=1e-6) * 100  # coefficient of variation %
imp_df_sorted = imp_df.sort_values('mean', ascending=False)

print("Feature Importances Across Seeds:")
print(f"{'Feature':25s} {'Mean':>8s} {'Std':>8s} {'CV%':>8s}  {'42':>7s} {'43':>7s} {'44':>7s} {'45':>7s} {'46':>7s} {'47':>7s}")
print("-" * 100)
for feat, row in imp_df_sorted.iterrows():
    per_seed = '  '.join([f'{row[s]:7.4f}' for s in SEEDS])
    print(f"{feat:25s} {row['mean']:8.4f} {row['std']:8.4f} {row['cv']:7.1f}%  {per_seed}")

# Rank stability: how often does each feature appear in the top 5?
print(f"\n{'='*60}")
print("TOP-5 FEATURE FREQUENCY (how often each feature lands in the top 5 by importance)")
print(f"{'='*60}")
top5_counts = {f: 0 for f in physical_features}
for seed in SEEDS:
    ranked = sorted(physical_features, key=lambda f: importances_per_seed[seed][physical_features.index(f)], reverse=True)
    for f in ranked[:5]:
        top5_counts[f] += 1
for f, count in sorted(top5_counts.items(), key=lambda x: -x[1]):
    bar = '#' * count
    print(f"  {f:25s} {count}/{len(SEEDS)} {bar}")

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── ROC-AUC across seeds ──
ax = axes[0]
ax.bar(range(len(SEEDS)), results['roc_auc'], color='steelblue', edgecolor='black', alpha=0.8)
ax.axhline(np.mean(results['roc_auc']), color='red', linestyle='--', label=f'Mean: {np.mean(results["roc_auc"]):.4f}')
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.set_ylabel('ROC-AUC')
ax.set_title('ROC-AUC by Seed')
ax.legend()
ax.set_ylim(0.7, 0.85)

# ── PR-AUC across seeds ──
ax = axes[1]
ax.bar(range(len(SEEDS)), results['pr_auc'], color='coral', edgecolor='black', alpha=0.8)
ax.axhline(np.mean(results['pr_auc']), color='red', linestyle='--', label=f'Mean: {np.mean(results["pr_auc"]):.4f}')
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.set_ylabel('PR-AUC')
ax.set_title('PR-AUC by Seed')
ax.legend()

# ── F1 across seeds ──
ax = axes[2]
ax.bar(range(len(SEEDS)), results['f1'], color='mediumpurple', edgecolor='black', alpha=0.8)
ax.axhline(np.mean(results['f1']), color='red', linestyle='--', label=f'Mean: {np.mean(results["f1"]):.4f}')
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.set_ylabel('F1')
ax.set_title('F1 by Seed')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
imp_data = [imp_df_sorted.loc[f, SEEDS].values for f in imp_df_sorted.index]
positions = range(len(imp_df_sorted))
ax.boxplot(imp_data, positions=positions, vert=True, widths=0.6)
ax.set_xticks(positions)
ax.set_xticklabels(imp_df_sorted.index, rotation=45, ha='right')
ax.set_ylabel('Importance')
ax.set_title(f'Feature Importance Distribution Across Seeds ({SEEDS[0]}–{SEEDS[-1]})')
plt.tight_layout()
plt.show()

We add three feature families: (1) multi-period/spectral Lomb-Scargle features derived from power distributions, not just peak statistics; (2) uncertainty-aware features separating measurement noise from real RV scatter; (3) enhanced RV-activity coupling. Then CV-tuned RF (RandomizedSearchCV) compared on phys18 vs the expanded set. No isotonic calibration.

In [ ]:
# Physics-preserving optimization: 5 enhanced + 13 Lomb-Scargle + 3 uncertainty = 21 new.
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)
from astropy.timeseries import LombScargle
warnings.filterwarnings('ignore')

N_FREQ = 500
F_MIN_DAYS = 1.0
LS_METHOD = 'fast'

# 8 original LS features (rv_ls_fap REMOVED — it's a noise p-value, not a planet discriminator)
# + 6 multi-period / power-concentration features from the same periodograms
_LS_ZERO_KEYS = ['rv_ls_peak_power', 'rv_ls_log_peak_period',
                 'rv_ls_mean_power', 'rhkp_ls_peak_power', 'rhkp_rv_power_ratio',
                 'halpha_ls_peak_power', 'halpha_rv_power_ratio', 'min_activity_rv_ratio',
                 'rv_top2_power_ratio', 'rv_top3_power_ratio',
                 'rv_power_conc_top10pct', 'rhkp_power_conc_top10pct',
                 'halpha_top2_power_ratio']

ENH_FEATURES = ['rv_rhkp_corr_abs', 'rv_halpha_corr_abs', 'rv_rhkp_partial_corr',
                'rv_rhkp_corr_std', 'rhkp_rv_ratio']
LS_FEATURES = list(_LS_ZERO_KEYS)  # 13
UNCERT_FEATURES = ['rv_excess_std', 'rv_snr', 'rv_weighted_amplitude']
DERIVED_FEATURES = ENH_FEATURES + LS_FEATURES + UNCERT_FEATURES  # 5 + 13 + 3 = 21

def compute_derived_features(group):
    """Per-star derived features: 5 enhanced + 13 Lomb-Scargle + 3 uncertainty = 21."""
    f = {}
    rv     = group['rv_centered'].values
    rhkp   = group['RHKp'].values
    halpha = group['Halpha'].values
    bjd    = group['bjd'].values
    rv_err = group['rv_err'].values
    n = len(rv)

    rv_s   = float(np.std(rv))     if n >= 2 else 0.0
    rv_s   = rv_s if rv_s == rv_s else 0.0
    rhkp_s = float(np.std(rhkp))   if n >= 2 else 0.0
    rhkp_s = rhkp_s if rhkp_s == rhkp_s else 0.0
    ha_s   = float(np.std(halpha)) if n >= 2 else 0.0
    ha_s   = ha_s if ha_s == ha_s else 0.0

    # ---- Uncertainty-aware features (3) ----
    rv_var = rv_s ** 2
    mean_rv_err2 = float(np.mean(rv_err ** 2)) if n > 0 else 0.0
    excess_var = max(rv_var - mean_rv_err2, 0.0)
    f['rv_excess_std'] = float(np.sqrt(excess_var))
    f['rv_snr'] = rv_s / float(np.mean(rv_err)) if n > 0 and float(np.mean(rv_err)) > 0 else 0.0

    # Weighted amplitude: fit sinusoid at dominant LS period, report fitted amplitude
    f['rv_weighted_amplitude'] = rv_s  # fallback; overwritten if LS runs

    # ---- 5 enhanced RV-activity coupling features ----
    f['rv_rhkp_corr_abs']   = abs(float(np.corrcoef(rv, rhkp)[0, 1])) if (rv_s > 0 and rhkp_s > 0) else 0.0
    f['rv_halpha_corr_abs'] = abs(float(np.corrcoef(rv, halpha)[0, 1])) if (rv_s > 0 and ha_s > 0) else 0.0
    # partial correlation (remove linear time trend)
    if n > 2 and rv_s > 0 and rhkp_s > 0:
        t = bjd - bjd.mean()
        A = np.column_stack([t, np.ones_like(t)])
        def _resid(y):
            coef = np.linalg.lstsq(A, y, rcond=None)[0]
            return y - (coef[0] * t + coef[1])
        rv_r, rhkp_r = _resid(rv), _resid(rhkp)
        std_rv_r, std_rhkp_r = float(np.std(rv_r)), float(np.std(rhkp_r))
        f['rv_rhkp_partial_corr'] = float(np.corrcoef(rv_r, rhkp_r)[0, 1]) if (std_rv_r > 0 and std_rhkp_r > 0) else 0.0
    else:
        f['rv_rhkp_partial_corr'] = 0.0
    # rolling-window correlation std
    w = min(20, n // 2)
    if w >= 5 and rv_s > 0 and rhkp_s > 0:
        step, corrs = max(1, w // 2), []
        for s in range(0, n - w + 1, step):
            if np.std(rhkp[s:s + w]) > 0 and np.std(rv[s:s + w]) > 0:
                c = float(np.corrcoef(rv[s:s + w], rhkp[s:s + w])[0, 1])
                if c == c and not np.isnan(c):
                    corrs.append(c)
        f['rv_rhkp_corr_std'] = float(np.std(corrs)) if len(corrs) >= 2 else 0.0
    else:
        f['rv_rhkp_corr_std'] = 0.0
    f['rhkp_rv_ratio'] = rhkp_s / rv_s if rv_s > 0 else 0.0

    # ---- 13 Lomb-Scargle periodogram features ----
    baseline = float(bjd.max() - bjd.min()) if n > 1 else 0.0
    p_max = max(baseline, 2.0) if baseline > 0 else 2.0
    freqs = np.linspace(1.0 / p_max, 1.0 / F_MIN_DAYS, N_FREQ)
    distinct_t = np.unique(np.round(bjd, 6)).size if n > 0 else 0
    can_run_ls = (n >= 4) and (rv_s > 0) and (baseline > 0) and (distinct_t >= 4)

    if can_run_ls:
        try:
            ls_rv = LombScargle(bjd, rv, dy=rv_err, normalization='standard', fit_mean=True)
            p_rv = ls_rv.power(freqs, method=LS_METHOD)
            pi = int(np.argmax(p_rv))
            rv_peak_power = float(p_rv[pi])
            rv_peak_period = float(1.0 / freqs[pi]) if freqs[pi] > 0 else 0.0

            # -- Weighted amplitude: fitted sine amplitude at LS peak period --
            x_fit = 2.0 * np.pi * freqs[pi] * (bjd - bjd.min())
            A_fit = np.column_stack([np.sin(x_fit), np.cos(x_fit)])
            coef = np.linalg.lstsq(A_fit, rv, rcond=None)[0]
            fitted_amp = float(np.sqrt(coef[0]**2 + coef[1]**2))
            f['rv_weighted_amplitude'] = fitted_amp

            ls_rhk = LombScargle(bjd, rhkp, normalization='standard', fit_mean=True)
            ls_hal = LombScargle(bjd, halpha, normalization='standard', fit_mean=True)
            p_rhk = ls_rhk.power(freqs, method=LS_METHOD)
            p_hal = ls_hal.power(freqs, method=LS_METHOD)

            rhkp_pow_at_rv = float(p_rhk[pi])
            halpha_pow_at_rv = float(p_hal[pi])
            rhkp_ratio   = rhkp_pow_at_rv / (rv_peak_power + 1e-9)
            halpha_ratio = halpha_pow_at_rv / (rv_peak_power + 1e-9)

            # -- Multi-period features --
            # Sorted peak indices (descending power)
            p_sorted_idx = np.argsort(p_rv)[::-1]
            top2_ratio = float(p_rv[p_sorted_idx[1]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 1 and p_rv[p_sorted_idx[0]] > 0 else 0.0
            top3_ratio = float(p_rv[p_sorted_idx[2]] / p_rv[p_sorted_idx[0]]) if len(p_sorted_idx) > 2 and p_rv[p_sorted_idx[0]] > 0 else 0.0

            # Power concentration: fraction of total power in top 10% of frequency bins
            n_top = max(1, len(p_rv) // 10)
            conc_top10pct = float(np.sum(np.sort(p_rv)[-n_top:]) / max(np.sum(p_rv), 1e-9))

            # Same for RHKp
            p_rhk_sorted = np.sort(p_rhk)
            rhk_conc_top10pct = float(np.sum(p_rhk_sorted[-n_top:]) / max(np.sum(p_rhk), 1e-9))

            # Halpha: top-2 ratio
            p_hal_sorted_idx = np.argsort(p_hal)[::-1]
            hal_top2_ratio = float(p_hal[p_hal_sorted_idx[1]] / p_hal[p_hal_sorted_idx[0]]) if len(p_hal_sorted_idx) > 1 and p_hal[p_hal_sorted_idx[0]] > 0 else 0.0

            f['rv_ls_peak_power']       = rv_peak_power
            f['rv_ls_log_peak_period']  = float(np.log10(max(rv_peak_period, 1e-9)))
            f['rv_ls_mean_power']       = float(np.mean(p_rv))
            f['rhkp_ls_peak_power']     = float(np.max(p_rhk))
            f['rhkp_rv_power_ratio']    = rhkp_ratio
            f['halpha_ls_peak_power']   = float(np.max(p_hal))
            f['halpha_rv_power_ratio']  = halpha_ratio
            f['min_activity_rv_ratio']  = min(rhkp_ratio, halpha_ratio)
            f['rv_top2_power_ratio']    = top2_ratio
            f['rv_top3_power_ratio']    = top3_ratio
            f['rv_power_conc_top10pct'] = conc_top10pct
            f['rhkp_power_conc_top10pct'] = rhk_conc_top10pct
            f['halpha_top2_power_ratio']  = hal_top2_ratio
        except Exception:
            for k in _LS_ZERO_KEYS:
                f[k] = 0.0
    else:
        for k in _LS_ZERO_KEYS:
            f[k] = 0.0

    f['has_exoplanets'] = int(group['has_exoplanets'].iloc[0])
    return pd.Series(f)

# Compute derived features per star
derived = (observations.groupby('star_name', sort=True)
           .apply(compute_derived_features, include_groups=False)
           .reset_index()
           .fillna(0))

# Merge with star_features
opt_features = star_features.merge(derived[['star_name'] + DERIVED_FEATURES],
                                   on='star_name', how='left').fillna(0)

# Define the expanded feature set name (e.g., phys39 or similar)
PHYS_NEW = list(physical_features) + DERIVED_FEATURES
FEATURE_SETS = {
    'phys18': (star_features, physical_features),
    'phys_new': (opt_features, PHYS_NEW),
}

print(f"Derived features: {len(DERIVED_FEATURES)}")
print(f"  Enhanced:   {len(ENH_FEATURES)}")
print(f"  LS:         {len(LS_FEATURES)}")
print(f"  Uncertainty:{len(UNCERT_FEATURES)}")
print(f"Total physical + derived: {len(PHYS_NEW)}  (was 32, now {len(PHYS_NEW)})")
print(f"Stars: {len(opt_features)}, pos={int(opt_features.has_exoplanets.sum())}, "
      f"neg={int((opt_features.has_exoplanets == 0).sum())}")

SEEDS = list(range(43, 48))
N_EST_FINAL = 1200
N_ITER, CV_FOLDS = 20, 4
param_dist = {
    'max_depth':         [8, 12, 16, 26, None],
    'min_samples_leaf':  [1, 2, 4, 8, 16],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2', 0.4, 0.7],
    'class_weight':      ['balanced', 'balanced_subsample', None],
}

opt = {'set': [], 'seed': [], 'roc_auc': [], 'pr_auc': [], 'f1': [], 'f05': [],
       'precision': [], 'recall': [], 'accuracy': [], 'threshold': [],
       'best_depth': [], 'best_msl': [], 'best_mss': [], 'best_mf': [],
       'best_cw': [], 'cv_pr_auc': []}
opt_importances = {}

BASELINE_PR = float(np.mean(results['pr_auc']))  # from the original multi-seed loop

sep = '#' * 72
for set_name, (frame, feats) in FEATURE_SETS.items():
    X_mat = frame[feats].values
    y_vec = frame['has_exoplanets'].values.astype(int)
    n_feat = len(feats)
    print(f"\n{sep}\n# FEATURE SET: {set_name}  ({n_feat} features)\n{sep}")

    for seed in SEEDS:
        idx = np.arange(len(frame))
        idx_train, idx_temp, _, _ = train_test_split(
            idx, y_vec, test_size=0.4, random_state=seed, stratify=y_vec)
        idx_val, idx_test, _, _ = train_test_split(
            idx_temp, y_vec[idx_temp], test_size=0.5,
            random_state=seed, stratify=y_vec[idx_temp])
        X_train, y_train = X_mat[idx_train], y_vec[idx_train]
        X_val,   y_val   = X_mat[idx_val],   y_vec[idx_val]
        X_test,  y_test  = X_mat[idx_test],  y_vec[idx_test]

        # CV-tune
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
        base = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=-1)
        search = RandomizedSearchCV(base, param_dist, scoring='average_precision',
                                    n_iter=N_ITER, cv=cv, n_jobs=-1,
                                    random_state=seed, refit=True, verbose=0)
        search.fit(X_train, y_train)
        bp = search.best_params_
        best_cv_pr = float(search.best_score_)
        del search

        # Refit with more trees
        rf = RandomForestClassifier(
            n_estimators=N_EST_FINAL, class_weight=bp['class_weight'],
            max_depth=bp['max_depth'], min_samples_leaf=bp['min_samples_leaf'],
            min_samples_split=bp['min_samples_split'], max_features=bp['max_features'],
            random_state=seed, n_jobs=-1)
        rf.fit(X_train, y_train)

        # Threshold on VAL (raw probs)
        val_probs = rf.predict_proba(X_val)[:, 1]
        test_probs = rf.predict_proba(X_test)[:, 1]
        vp, vr, vt = precision_recall_curve(y_val, val_probs)
        vf1 = 2 * vp * vr / (vp + vr + 1e-8); bi = int(np.argmax(vf1))
        thr = float(vt[bi]) if bi < len(vt) else 0.5

        # Evaluate on TEST
        roc = roc_auc_score(y_test, test_probs)
        pr  = average_precision_score(y_test, test_probs)
        tp = (test_probs >= thr).astype(int)
        cm = confusion_matrix(y_test, tp); tn, fp, fn, tpc = cm.ravel()
        prc = tpc / (tpc + fp) if (tpc + fp) > 0 else 0.0
        rec = tpc / (tpc + fn) if (tpc + fn) > 0 else 0.0
        f1 = f1_score(y_test, tp, zero_division=0)
        f05 = fbeta_score(y_test, tp, beta=0.5, zero_division=0)
        acc = accuracy_score(y_test, tp)

        opt['set'].append(set_name); opt['seed'].append(seed)
        opt['roc_auc'].append(roc); opt['pr_auc'].append(pr)
        opt['f1'].append(f1); opt['f05'].append(f05)
        opt['precision'].append(prc); opt['recall'].append(rec)
        opt['accuracy'].append(acc); opt['threshold'].append(thr)
        opt['best_depth'].append(str(bp['max_depth']))
        opt['best_msl'].append(bp['min_samples_leaf'])
        opt['best_mss'].append(bp['min_samples_split'])
        opt['best_mf'].append(str(bp['max_features']))
        opt['best_cw'].append(str(bp['class_weight']))
        opt['cv_pr_auc'].append(best_cv_pr)
        opt_importances[(set_name, seed)] = rf.feature_importances_

        print(f"  [{set_name} s{seed}] PR-AUC={pr:.4f}  "
              f"ROC={roc:.4f}  F1={f1:.4f} (thr={thr:.3f})  "
              f"d={bp['max_depth']} msl={bp['min_samples_leaf']} "
              f"mf={bp['max_features']} cw={bp['class_weight']} "
              f"CV_PrAUC={best_cv_pr:.4f}")

opt_df = pd.DataFrame(opt)
print('\n=== Per-seed test metrics ===')
print(opt_df[['set', 'seed', 'pr_auc', 'roc_auc', 'f1', 'f05',
              'precision', 'recall', 'threshold', 'cv_pr_auc']].to_string(index=False))

print(f"\n=== Aggregate (mean +/- std, n={len(SEEDS)} outer seeds) ===")
for set_name in FEATURE_SETS:
    sub = opt_df[opt_df['set'] == set_name]
    print(f"\n[{set_name}]")
    for m in ['pr_auc', 'roc_auc', 'f1', 'f05', 'precision', 'recall']:
        v = sub[m].values
        print(f"  {m:11s}: {v.mean():.4f} +/- {v.std(ddof=1):.4f}  "
              f"(min={v.min():.4f}, max={v.max():.4f})")

# Lift vs baseline
print(f"\n=== Lift vs original untuned phys18 RF (mean PR-AUC = {BASELINE_PR:.4f}, target 0.70) ===")
for set_name in FEATURE_SETS:
    mv = float(opt_df[opt_df['set'] == set_name]['pr_auc'].mean())
    flag = '  ** TARGET BREACHED **' if mv >= 0.70 else ''
    print(f"  {set_name:10s}: mean PR-AUC = {mv:.4f}  (delta = {mv - BASELINE_PR:+.4f}){flag}")

# Feature importances
print("\n=== Feature importances (phys_new, mean over seeds) ===")
full_imp_arrs = [opt_importances[('phys_new', s)].astype(float) for s in SEEDS]
imp2 = pd.DataFrame(np.vstack(full_imp_arrs).T, index=PHYS_NEW, columns=SEEDS)
imp2['mean'] = imp2[SEEDS].mean(axis=1)
for feat, row in imp2.sort_values('mean', ascending=False).iterrows():
    print(f"  {feat:30s} {row['mean']:.4f}")

# Best hyperparams
print(f"\n=== Best hyperparameters per seed (phys_new) ===")
sub = opt_df[opt_df['set'] == 'phys_new'][['seed', 'best_depth', 'best_msl',
          'best_mss', 'best_mf', 'best_cw', 'pr_auc', 'cv_pr_auc']]
print(sub.to_string(index=False))